# Can we predict late delivery?
Protocol: `protocol.yaml` (temporal holdout from `holdout.csv`, 5-split walk-forward CV on training rows, choose by mean CV ROC AUC, score the holdout once).

Feature decision: `claim_paid_eur` is compensation paid on a claim settled for the consignment, i.e. recorded after the outcome, so it is **not** available at prediction time and is excluded (it would leak the target). `consignment` (identifier) and `dispatched_on` (split column) are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import roc_auc_score
from sklearn.base import clone

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='fae5c02170970fe92fcadaee6d1109bb1d459c9dc033dfc487bb2c823d0f6736'
df=pd.read_csv('data.csv'); df['dispatched_on']=pd.to_datetime(df['dispatched_on'])
hold_idx=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==8051 and len(hold_idx)==1610
train=df.drop(index=hold_idx).sort_values('dispatched_on', kind='stable')   # index = 0-based row position in data.csv
hold=df.loc[np.sort(hold_idx)]
print(len(train), len(hold), train.dispatched_on.max(), hold.dispatched_on.min())
print(train.late_delivery.mean(), hold.late_delivery.mean())

In [ ]:
CAT=['carrier']; NUM=['distance_km','weight_kg','parcels']; FEATURES=CAT+NUM
y_tr=train['late_delivery']; X_tr=train[FEATURES]
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
# all candidates named before any scoring
candidates={
 'baseline_prior': Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'rf_depth6': Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,max_depth=6,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hgb_small': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
}
print(list(candidates))

In [ ]:
tscv=TimeSeriesSplit(n_splits=5)
pos=train.index.to_numpy()   # data.csv row positions, time-ordered
folds=[(pos[a].tolist(),pos[b].tolist()) for a,b in tscv.split(train)]
cv_scores={}
for name,est in candidates.items():
    s=[]
    for tr,va in folds:
        m=clone(est).fit(df.loc[tr,FEATURES],df.loc[tr,'late_delivery'])   # preprocessing is inside the pipeline, fit per fold
        s.append(roc_auc_score(df.loc[va,'late_delivery'],m.predict_proba(df.loc[va,FEATURES])[:,1]))
    cv_scores[name]=s
cv=pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); print(cv.round(4))
chosen=cv['mean'].idxmax(); print('chosen:',chosen)

In [ ]:
final=clone(candidates[chosen]).fit(X_tr,y_tr)
auc=roc_auc_score(hold['late_delivery'],final.predict_proba(hold[FEATURES])[:,1])
print(f'Holdout ROC AUC ({chosen}): {auc:.4f}')   # scored once
evidence=dict(features=FEATURES, candidates=list(candidates),
  cv_folds=[[a,b] for a,b in folds],
  cv_scores=cv_scores, chosen=chosen,
  holdout_rows=[int(i) for i in hold.index],
  holdout_scores={chosen:auc})
json.dump(evidence,open('evidence.json','w'))